# NLP Task 5 – Fine-Tuning DistilBERT for POS Tagging & Chunking






## Install Required Libraries

In [ ]:
!pip install transformers datasets seqeval evaluate accelerate -q

import nltk
# CoNLL-2000 corpus: POS tags + chunk tags, available via NLTK with no HF script dependency
nltk.download('conll2000', quiet=True)
print('Packages ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
Packages ready


## Import Libraries

In [ ]:

import numpy as np
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import conll2000

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
import evaluate

print('All libraries imported successfully!')

All libraries imported successfully!



## Task 1 – Dataset Selection (10%)

We use the **CoNLL-2000** dataset loaded via **NLTK**.

It contains:
- **POS tags** – Penn Treebank style (NN, VBZ, JJ, IN, DT …)
- **Chunk tags** – BIO phrase tags (B-NP, I-NP, B-VP, O …)

> **Why NLTK and not HuggingFace Hub?**
> Both `conll2003` and every mirror on HF Hub still include a
> legacy Python loading script (`conll2003.py`). The current `datasets`
> library raises `RuntimeError: Dataset scripts are no longer supported`
> when it finds such a file — even with `trust_remote_code=True` removed.
> NLTK ships the CoNLL-2000 data as plain text with no script, so it
> works in every environment.

In [ ]:
# iob_sents() returns a list of sentences.
# Each sentence is a list of (word, pos_tag, chunk_tag) tuples.
raw_train = list(conll2000.iob_sents('train.txt'))
raw_test  = list(conll2000.iob_sents('test.txt'))

print(f'Train sentences : {len(raw_train):,}')
print(f'Test  sentences : {len(raw_test):,}')
print('\nFirst sentence sample (word, POS, chunk):')
for tok in raw_train[0][:8]:
    print(' ', tok)

Train sentences : 8,936
Test  sentences : 2,012

First sentence sample (word, POS, chunk):
  ('Confidence', 'NN', 'B-NP')
  ('in', 'IN', 'B-PP')
  ('the', 'DT', 'B-NP')
  ('pound', 'NN', 'I-NP')
  ('is', 'VBZ', 'B-VP')
  ('widely', 'RB', 'I-VP')
  ('expected', 'VBN', 'I-VP')
  ('to', 'TO', 'I-VP')


In [ ]:
# Collect every unique label that appears in the full corpus
all_pos_labels   = sorted({pos   for sent in raw_train + raw_test for _, pos,   _ in sent})
all_chunk_labels = sorted({chunk for sent in raw_train + raw_test for _, _, chunk in sent})

# String -> integer mappings (model expects integer IDs)
pos2id   = {l: i for i, l in enumerate(all_pos_labels)}
chunk2id = {l: i for i, l in enumerate(all_chunk_labels)}

POS_LABEL_LIST   = all_pos_labels
CHUNK_LABEL_LIST = all_chunk_labels

print(f'Unique POS labels   : {len(POS_LABEL_LIST)}')
print(f'Unique Chunk labels : {len(CHUNK_LABEL_LIST)}')
print('\nPOS labels  :', POS_LABEL_LIST)
print('Chunk labels:', CHUNK_LABEL_LIST)

Unique POS labels   : 44
Unique Chunk labels : 23

POS labels  : ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']
Chunk labels: ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']


In [ ]:
def sents_to_dict(sentences):
    """Convert list-of-tuple sentences into a column-oriented dict with integer label IDs."""
    tokens_col, pos_col, chunk_col = [], [], []
    for sent in sentences:
        tokens_col.append([w for w, _, _ in sent])
        pos_col.append(   [pos2id[p]   for _, p, _ in sent])
        chunk_col.append( [chunk2id[c] for _, _, c in sent])
    return {'tokens': tokens_col, 'pos_tags': pos_col, 'chunk_tags': chunk_col}

train_dict = sents_to_dict(raw_train)
test_dict  = sents_to_dict(raw_test)

# Split the original test set 50/50 to get a validation split
full_test = Dataset.from_dict(test_dict)
splits    = full_test.train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    'train'     : Dataset.from_dict(train_dict),
    'validation': splits['train'],
    'test'      : splits['test'],
})

print(dataset)

s = dataset['train'][0]
print('\ntokens     :', s['tokens'][:6])
print('pos_tags   :', s['pos_tags'][:6], '->', [POS_LABEL_LIST[i] for i in s['pos_tags'][:6]])
print('chunk_tags :', s['chunk_tags'][:6], '->', [CHUNK_LABEL_LIST[i] for i in s['chunk_tags'][:6]])

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 8936
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 1006
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 1006
    })
})

tokens     : ['Confidence', 'in', 'the', 'pound', 'is', 'widely']
pos_tags   : [18, 13, 10, 18, 38, 26] -> ['NN', 'IN', 'DT', 'NN', 'VBZ', 'RB']
chunk_tags : [5, 6, 5, 16, 10, 21] -> ['B-NP', 'B-PP', 'B-NP', 'I-NP', 'B-VP', 'I-VP']


---
## Task 2 – Data Preprocessing (15%)

BERT uses **WordPiece tokenization** which may split a single word into multiple sub-tokens.
We must **align labels** so that:
- The **first sub-token** of a word gets the real label.
- **Continuation sub-tokens** and **special tokens** ([CLS], [SEP]) get `-100` so they are ignored by the loss.

In [ ]:
# Configuration
MODEL_NAME = 'distilbert-base-uncased'  # 40% smaller than BERT, ~97% of its accuracy

TASK = 'chunking'   # Change to 'pos' to train POS tagger instead

LABEL_LIST = CHUNK_LABEL_LIST if TASK == 'chunking' else POS_LABEL_LIST
LABEL_KEY  = 'chunk_tags'     if TASK == 'chunking' else 'pos_tags'

print(f'Task   : {TASK.upper()}')
print(f'Labels : {LABEL_LIST}')

Task   : CHUNKING
Labels : ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: distilbert-base-uncased


In [ ]:
def tokenize_and_align_labels(examples):
    """
    Tokenizes word-level inputs and aligns token-level labels.

    Steps:
      1. Tokenize with is_split_into_words=True (input words are already split).
      2. word_ids() maps every sub-token back to its original word index.
      3. First sub-token of a word -> real label.
      4. Continuation sub-tokens and special tokens -> -100 (ignored by loss).
    """
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )

    all_labels = []
    for i, label_ids in enumerate(examples[LABEL_KEY]):
        word_ids       = tokenized_inputs.word_ids(batch_index=i)
        prev_word_idx  = None
        aligned_labels = []

        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)                  # [CLS] / [SEP]
            elif word_idx != prev_word_idx:
                aligned_labels.append(label_ids[word_idx])  # First sub-token
            else:
                aligned_labels.append(-100)                  # Continuation sub-token
            prev_word_idx = word_idx

        all_labels.append(aligned_labels)

    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

print('tokenize_and_align_labels() defined')

tokenize_and_align_labels() defined


In [ ]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names,
)

print('Tokenization & label alignment complete!')
print(tokenized_datasets)

# Inspect one preprocessed example
ex = tokenized_datasets['train'][0]
print('\nSub-tokens :', tokenizer.convert_ids_to_tokens(ex['input_ids']))
print('Labels     :', ex['labels'])

Map:   0%|          | 0/8936 [00:00<?, ? examples/s]

Map:   0%|          | 0/1006 [00:00<?, ? examples/s]

Map:   0%|          | 0/1006 [00:00<?, ? examples/s]

Tokenization & label alignment complete!
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8936
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1006
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1006
    })
})

Sub-tokens : ['[CLS]', 'confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'september', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'july', 'and', 'august', "'", 's', 'near', '-', 'record', 'deficit', '##s', '.', '[SEP]']
Labels     : [-100, 5, 6, 5, 16, 10, 21, 21, 21, 21, 5, 16, 16, 8, 5, 16, 6, 5, 22, 0, 6, 5, 5, 22, 10, 21, 21, 5, 16, 16, 6, 5, 16, 16, 5, -100, 16, -100, -100, 16, -100


## Task 3 – Model Setup (15%)

We use `AutoModelForTokenClassification` from HuggingFace with:
- `num_labels` = number of unique label classes
- `id2label` (int → string) for readable inference
- `label2id` (string → int) required by the model

In [ ]:
id2label = {i: label for i, label in enumerate(LABEL_LIST)}
label2id = {label: i for i, label in enumerate(LABEL_LIST)}

print('id2label:', id2label)
print('label2id:', label2id)

id2label: {0: 'B-ADJP', 1: 'B-ADVP', 2: 'B-CONJP', 3: 'B-INTJ', 4: 'B-LST', 5: 'B-NP', 6: 'B-PP', 7: 'B-PRT', 8: 'B-SBAR', 9: 'B-UCP', 10: 'B-VP', 11: 'I-ADJP', 12: 'I-ADVP', 13: 'I-CONJP', 14: 'I-INTJ', 15: 'I-LST', 16: 'I-NP', 17: 'I-PP', 18: 'I-PRT', 19: 'I-SBAR', 20: 'I-UCP', 21: 'I-VP', 22: 'O'}
label2id: {'B-ADJP': 0, 'B-ADVP': 1, 'B-CONJP': 2, 'B-INTJ': 3, 'B-LST': 4, 'B-NP': 5, 'B-PP': 6, 'B-PRT': 7, 'B-SBAR': 8, 'B-UCP': 9, 'B-VP': 10, 'I-ADJP': 11, 'I-ADVP': 12, 'I-CONJP': 13, 'I-INTJ': 14, 'I-LST': 15, 'I-NP': 16, 'I-PP': 17, 'I-PRT': 18, 'I-SBAR': 19, 'I-UCP': 20, 'I-VP': 21, 'O': 22}


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,   # Replaces the pre-trained classification head
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model    : {MODEL_NAME}')
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model    : distilbert-base-uncased
Total params     : 66,380,567
Trainable params : 66,380,567



## Task 4 – Model Training (20%)

Using HuggingFace `Trainer` with:
- Learning rate: `2e-5`
- Epochs: `3`
- Batch size: `16`

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load('seqeval')

def compute_metrics(p):
    """
    Takes (logits, true_labels), converts to string label sequences,
    and computes seqeval precision / recall / F1.
    """
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)  # logits -> class indices

    true_predictions = [
        [LABEL_LIST[pred] for pred, lbl in zip(pr, la) if lbl != -100]
        for pr, la in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[lbl] for lbl in la if lbl != -100]
        for la in labels
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall'   : results['overall_recall'],
        'f1'       : results['overall_f1'],
        'accuracy' : results['overall_accuracy'],
    }

print('Data collator and metrics function ready')

Data collator and metrics function ready


In [ ]:
training_args = TrainingArguments(
    output_dir=f'./results_{TASK}',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    push_to_hub=False,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer ready')

Trainer ready


In [ ]:
# TIP: Enable GPU in Colab → Runtime > Change runtime type > GPU T4
# Training time: ~10-15 min (GPU) vs ~60+ min (CPU)
print(f'Starting fine-tuning DistilBERT for {TASK.upper()} ...')
trainer.train()
print('Training complete!')

Starting fine-tuning DistilBERT for CHUNKING ...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.125438,0.118770,0.946880,0.958879,0.952842,0.970573
2,0.091160,0.104533,0.953046,0.962403,0.957702,0.974008
3,0.070720,0.103764,0.957749,0.962571,0.960154,0.974644


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training complete!


---
## Task 5 – Evaluation (15%)

Using `seqeval` which computes:
- **Precision** – Of all predicted chunks, how many are correct?
- **Recall** – Of all true chunks, how many did the model find?
- **F1 Score** – Harmonic mean of Precision and Recall

In [ ]:
results = trainer.evaluate(eval_dataset=tokenized_datasets['test'])

print('\nTest Set Evaluation Results')
print('='*40)
print(f"  Precision : {results['eval_precision']:.4f}")
print(f"  Recall    : {results['eval_recall']:.4f}")
print(f"  F1 Score  : {results['eval_f1']:.4f}")
print(f"  Accuracy  : {results['eval_accuracy']:.4f}")
print('='*40)


Test Set Evaluation Results
  Precision : 0.9541
  Recall    : 0.9588
  F1 Score  : 0.9565
  Accuracy  : 0.9731


In [ ]:
# Per-class breakdown
preds, labs, _ = trainer.predict(tokenized_datasets['test'])
preds = np.argmax(preds, axis=2)

true_preds = [
    [LABEL_LIST[p] for p, l in zip(pr, la) if l != -100]
    for pr, la in zip(preds, labs)
]
true_labs = [
    [LABEL_LIST[l] for l in la if l != -100]
    for la in labs
]

detailed = seqeval.compute(predictions=true_preds, references=true_labs)
print(f"{'Label':<15} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print('-'*57)
for key, val in sorted(detailed.items()):
    if isinstance(val, dict):
        print(f"{key:<15} {val['precision']:>10.4f} {val['recall']:>10.4f} "
              f"{val['f1']:>10.4f} {val['number']:>10}")

Label            Precision     Recall         F1    Support
---------------------------------------------------------
ADJP                0.7811     0.7441     0.7621        211
ADVP                0.8264     0.8400     0.8331        425
CONJP               0.5000     0.2500     0.3333          4
INTJ                0.0000     0.0000     0.0000          1
LST                 0.0000     0.0000     0.0000          4
NP                  0.9632     0.9646     0.9639       6212
PP                  0.9746     0.9876     0.9810       2410
PRT                 0.7833     0.8868     0.8319         53
SBAR                0.9203     0.9373     0.9287        271
VP                  0.9559     0.9620     0.9590       2345


---
## Task 6 – Inference on Custom Sentences (10%)

In [ ]:
ner_pipe = pipeline(
    'token-classification',
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy='simple',  # Merge consecutive same-label sub-tokens
)

def predict_and_display(sentence):
    output = ner_pipe(sentence)
    print(f'\nInput  : {sentence}')
    print(f'Task   : {TASK.upper()}')
    print('-'*55)
    print(f"{'Token':<20} {'Label':<15} {'Score':>8}")
    print('-'*55)
    for tok in output:
        print(f"{tok['word']:<20} {tok['entity_group']:<15} {tok['score']:>8.4f}")
    print('-'*55)

# Canonical example from the assignment spec + additional sentences
for sent in [
    'John works at Google in California.',
    'The quick brown fox jumps over the lazy dog.',
    'Apple released a new iPhone with advanced AI features.',
    'She is reading an interesting book about machine learning.',
]:
    predict_and_display(sent)


Input  : John works at Google in California.
Task   : CHUNKING
-------------------------------------------------------
Token                Label              Score
-------------------------------------------------------
john                 NP                0.9982
works                VP                0.9983
at                   PP                0.9974
google               NP                0.9977
in                   PP                0.9976
california           NP                0.9980
-------------------------------------------------------

Input  : The quick brown fox jumps over the lazy dog.
Task   : CHUNKING
-------------------------------------------------------
Token                Label              Score
-------------------------------------------------------
the quick brown fox  NP                0.9975
jumps                VP                0.9983
over                 PP                0.9783
the lazy dog         NP                0.9981
-------------------------------


## Task 7 – Comparison: POS Tagging vs Chunking (10%)

In [ ]:
def train_for_task(task_name):
    """Generic training + evaluation for a given task ('pos' or 'chunking')."""
    lbl_key  = 'chunk_tags' if task_name == 'chunking' else 'pos_tags'
    lbl_list = CHUNK_LABEL_LIST if task_name == 'chunking' else POS_LABEL_LIST
    i2l = {i: l for i, l in enumerate(lbl_list)}
    l2i = {l: i for i, l in enumerate(lbl_list)}

    def _tok(examples):
        enc = tokenizer(examples['tokens'], truncation=True,
                        is_split_into_words=True, max_length=128)
        all_labels = []
        for idx, lids in enumerate(examples[lbl_key]):
            wids, prev, row = enc.word_ids(batch_index=idx), None, []
            for wid in wids:
                row.append(-100 if wid is None or wid == prev else lids[wid])
                prev = wid
            all_labels.append(row)
        enc['labels'] = all_labels
        return enc

    tok_ds = dataset.map(_tok, batched=True, remove_columns=dataset['train'].column_names)

    _model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=len(lbl_list),
        id2label=i2l, label2id=l2i, ignore_mismatched_sizes=True)

    _seqeval = evaluate.load('seqeval')

    def _metrics(p):
        pr, la = p
        pr = np.argmax(pr, axis=2)
        tp = [[lbl_list[p_] for p_, l_ in zip(r_pr, r_la) if l_ != -100] for r_pr, r_la in zip(pr, la)]
        tl = [[lbl_list[l_] for l_ in r_la if l_ != -100] for r_la in la]
        r  = _seqeval.compute(predictions=tp, references=tl)
        return {'precision': r['overall_precision'], 'recall': r['overall_recall'], 'f1': r['overall_f1']}

    _trainer = Trainer(
        model=_model,
        args=TrainingArguments(
            output_dir=f'./results_{task_name}', eval_strategy='epoch',
            save_strategy='epoch', learning_rate=2e-5,
            per_device_train_batch_size=16, per_device_eval_batch_size=16,
            num_train_epochs=3, weight_decay=0.01,
            load_best_model_at_end=True, metric_for_best_model='f1', report_to='none'),
        train_dataset=tok_ds['train'], eval_dataset=tok_ds['validation'],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=_metrics,
    )
    _trainer.train()
    res = _trainer.evaluate(eval_dataset=tok_ds['test'])
    return {k: round(v, 4) for k, v in res.items() if k in ['eval_precision', 'eval_recall', 'eval_f1']}

print('train_for_task() ready')

train_for_task() ready


In [ ]:
print('Training POS model ...')
pos_results = train_for_task('pos')
print('POS results:', pos_results)

Training POS model ...


Map:   0%|          | 0/8936 [00:00<?, ? examples/s]

Map:   0%|          | 0/1006 [00:00<?, ? examples/s]

Map:   0%|          | 0/1006 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


POS results: {'eval_precision': 0.9618, 'eval_recall': 0.9645, 'eval_f1': 0.9632}


In [ ]:
print('Training Chunking model ...')
chunk_results = train_for_task('chunking')
print('Chunking results:', chunk_results)

Training Chunking model ...


Map:   0%|          | 0/8936 [00:00<?, ? examples/s]

Map:   0%|          | 0/1006 [00:00<?, ? examples/s]

Map:   0%|          | 0/1006 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Chunking results: {'eval_precision': 0.9489, 'eval_recall': 0.9581, 'eval_f1': 0.9535}


In [ ]:
print('\n' + '='*55)
print('  COMPARISON: POS Tagging  vs  Chunking')
print('='*55)
print(f"{'Metric':<15} {'POS Tagging':>15} {'Chunking':>15}")
print('-'*47)
for metric, key in [('Precision','eval_precision'),('Recall','eval_recall'),('F1 Score','eval_f1')]:
    print(f"{metric:<15} {str(pos_results.get(key,'N/A')):>15} {str(chunk_results.get(key,'N/A')):>15}")
print('='*55)
print('''
Observations:
  POS Tagging  -> Tags individual words with grammatical roles (NN, VB, JJ).
               -> Simpler; higher F1 (~96-98%) with DistilBERT.

  Chunking     -> Groups tokens into phrases using BIO encoding (B-NP, I-NP).
               -> Harder; typically F1 ~92-95% with DistilBERT.
               -> Requires model to understand phrase-level boundaries.
''')


  COMPARISON: POS Tagging  vs  Chunking
Metric              POS Tagging        Chunking
-----------------------------------------------
Precision                0.9618          0.9489
Recall                   0.9645          0.9581
F1 Score                 0.9632          0.9535

Observations:
  POS Tagging  -> Tags individual words with grammatical roles (NN, VB, JJ).
               -> Simpler; higher F1 (~96-98%) with DistilBERT.

  Chunking     -> Groups tokens into phrases using BIO encoding (B-NP, I-NP).
               -> Harder; typically F1 ~92-95% with DistilBERT.
               -> Requires model to understand phrase-level boundaries.



---
## Task 8 – Report / Blog (5%)

### Differences Between POS Tagging and Chunking

| Feature | POS Tagging | Chunking |
|---|---|---|
| Granularity | Word-level | Phrase-level |
| Output | One tag per token (NN, VB, JJ …) | BIO tags (B-NP, I-VP, O …) |
| Complexity | Lower | Higher |
| Use Case | Grammar checking, morphology | Information extraction, QA |
| Dataset | Penn Treebank, UD | CoNLL-2000 |
| Typical BERT F1 | ~96-98% | ~92-95% |

---

### Key Concepts Learned

1. **Token Classification** – Assigning a label to every sub-token independently.
2. **WordPiece Tokenization** – BERT splits words; label alignment with `-100` masking is essential.
3. **Label Alignment** – First sub-token gets the real label; continuations and special tokens get `-100`.
4. **BIO Tagging Scheme** – B = Beginning of phrase, I = Inside, O = Outside.
5. **seqeval Metric** – Evaluates at chunk level (entire span must match for a true positive).
6. **Fine-Tuning** – DistilBERT pre-training provides rich language understanding; we only adapt the head.

---

### Challenges Faced

1. **HuggingFace Script Deprecation** – `conll2003` on HF Hub includes a legacy script now blocked by the `datasets` library. Resolved by switching to NLTK's `conll2000` corpus and building a `DatasetDict` manually.
2. **Sub-token Label Alignment** – Multi-sub-token words require careful `-100` masking.
3. **Class Imbalance** – The `O` chunk tag dominates the data.
4. **seqeval Strictness** – Token accuracy can be misleadingly high; seqeval is the correct metric.

---

### Insights & Observations

- BERT's contextual embeddings give a strong starting point for token-level tasks.
- DistilBERT is 40% smaller than BERT with only ~3% accuracy drop — ideal for resource-constrained environments.
- POS tagging converges faster because class boundaries are simpler.
- `DataCollatorForTokenClassification` handles variable-length padding efficiently within each batch.
- 3 fine-tuning epochs are sufficient to reach strong performance thanks to rich BERT pre-training.

---
## Save & Reload Model

In [ ]:
SAVE_DIR = f'./finetuned_distilbert_{TASK}'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model saved to {SAVE_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./finetuned_distilbert_chunking


In [ ]:
# Load from disk and run final inference check
saved_tok   = AutoTokenizer.from_pretrained(SAVE_DIR)
saved_model = AutoModelForTokenClassification.from_pretrained(SAVE_DIR)

inf_pipe = pipeline('token-classification', model=saved_model,
                    tokenizer=saved_tok, aggregation_strategy='simple')

sentence = 'John works at Google in California.'
out = inf_pipe(sentence)

print(f'Input : {sentence}')
print(f"{'Token':<20} {'Tag':<15} {'Score':>8}")
print('-'*47)
for t in out:
    print(f"{t['word']:<20} {t['entity_group']:<15} {t['score']:>8.4f}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Input : John works at Google in California.
Token                Tag                Score
-----------------------------------------------
john                 NP                0.9982
works                VP                0.9983
at                   PP                0.9974
google               NP                0.9977
in                   PP                0.9976
california           NP                0.9980
